# GPT-OSS: Pre-Training from Scratch

This notebook contains the complete pipeline to pre-train a Nano GPT-OSS model from scratch on the TinyStories dataset.

**Hardware:** NVIDIA H100 GPU  
**Dataset:** TinyStories (2M stories)  
**Architecture:** GPT-OSS (GQA + RoPE + Sliding Window + MoE + SwiGLU + Attention Sinks)  
**Expected Training Time:** ~2-3 hours on H100  
**Estimated Cost:** < $10

---

### Pipeline Overview
```
1. Load TinyStories dataset (2M short stories)
2. Tokenize with Harmony tokenizer (subword BPE, vocab=201088)
3. Create input-output pairs (input shifted right by 1 = target)
4. Build GPT-OSS architecture (RMSNorm + GQA + RoPE + Sliding + Sinks + MoE + SwiGLU)
5. Train with AdamW + Cosine Annealing LR schedule
6. Save weights to disk / Google Drive
```

## 1. Setup & Dependencies

In [ ]:
!pip install -q torch datasets tiktoken wandb tqdm ipywidgets

In [ ]:
import os
import gc
import json
import math
import time
from dataclasses import dataclass, asdict

import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LinearLR, SequentialLR, CosineAnnealingLR

import tiktoken
from datasets import load_dataset
from tqdm.notebook import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

os.makedirs("checkpoints", exist_ok=True)

## 2. Tokenizer (O200K Harmony)

GPT-OSS uses a **subword tokenizer** based on Byte Pair Encoding (BPE):
- Starts with OpenAI's O200K base (~200K BPE merge rules)
- Adds special tokens for chat roles, tool calls, and reasoning channels
- Final vocab size: **201,088**

Why subword over word-level? No OOV problem, manageable vocab, preserves morphological meaning.  
Why subword over character-level? Preserves word semantics, doesn't blow up sequence length.

In [ ]:
def get_tokenizer():
    """Build the O200K Harmony tokenizer used by GPT-OSS.
    
    Harmony extends the O200K base with special tokens for:
    - Chat formatting (<|start|>, <|end|>, <|message|>)
    - Tool/function calling (<|call|>, <|return|>, <|constrain|>)
    - Reasoning channels (<|channel|> for analysis/commentary/final)
    
    For plain text (TinyStories), these aren't used — tokenization is 
    identical to O200K base. They matter for conversational pre-training.
    """
    o200k_base = tiktoken.get_encoding("o200k_base")
    tokenizer = tiktoken.Encoding(
        name="o200k_harmony",
        pat_str=o200k_base._pat_str,            # Same regex for splitting text
        mergeable_ranks=o200k_base._mergeable_ranks,  # Same BPE merge rules
        special_tokens={
            **o200k_base._special_tokens,        # Inherit all base special tokens
            "<|startoftext|>": 199998,           # Document boundary
            "<|endoftext|>": 199999,             # Document boundary
            "<|reserved_200000|>": 200000,
            "<|reserved_200001|>": 200001,
            "<|return|>": 200002,                # Function return marker
            "<|constrain|>": 200003,             # Output constraint
            "<|reserved_200004|>": 200004,
            "<|channel|>": 200005,               # analysis/commentary/final
            "<|start|>": 200006,                 # Block start
            "<|end|>": 200007,                   # Block end  
            "<|message|>": 200008,               # Chat message boundary
            "<|reserved_200009|>": 200009,
            "<|reserved_200010|>": 200010,
            "<|reserved_200011|>": 200011,
            "<|call|>": 200012,                  # Tool/function call
        } | {
            # Reserve IDs 200013-201087 for future expansion
            f"<|reserved_{i}|>": i for i in range(200013, 201088)
        },
    )
    return tokenizer

tokenizer = get_tokenizer()
print(f"Vocabulary size: {tokenizer.n_vocab}")

## 3. Data Loading & Preprocessing

**TinyStories** — 2M synthetic short stories (3-4 year old vocabulary).  
The model must learn English grammar and semantics purely from predicting the next token.

Pipeline: `Raw stories → Join into one long text → Tokenize → Slice into fixed-length windows → Create (input, target) pairs`

In [ ]:
# H100 has 80GB VRAM — we can use larger batches than A40 (48GB)
BATCH_SIZE = 8       # Number of sequences processed in parallel
CONTEXT_LEN = 4096   # Max tokens the model sees at once (its "attention window")

print("Loading TinyStories dataset from HuggingFace...")
dataset = load_dataset("roneneldan/TinyStories")
print(f"Train stories: {len(dataset['train']):,}")
print(f"Validation stories: {len(dataset['validation']):,}")

In [ ]:
# Join all stories into one continuous text stream
# The model doesn't see story boundaries — it learns them from patterns
print("Joining text...")
train_text = " ".join([ex["text"] for ex in tqdm(dataset['train'], desc="Train")])
val_text = " ".join([ex["text"] for ex in tqdm(dataset['validation'], desc="Val")])

print(f"\nTrain text length: {len(train_text):,} characters")
print(f"Val text length: {len(val_text):,} characters")

In [ ]:
# Tokenize: convert characters → token IDs using BPE
# "Once upon a time" → [10234, 5621, 264, 892]
print("Tokenizing (this takes a few minutes)...")
train_tokens = tokenizer.encode(train_text)
val_tokens = tokenizer.encode(val_text)

print(f"Train tokens: {len(train_tokens):,}")
print(f"Val tokens: {len(val_tokens):,}")
print(f"Total tokens: {len(train_tokens) + len(val_tokens):,}")

# Free memory — we no longer need raw text
del dataset, train_text, val_text
gc.collect()

In [ ]:
class TextDataset(Dataset):
    """Creates (input, target) pairs for next-token prediction.
    
    The key insight: target = input shifted right by 1 position.
    
    Example (context_len=5):
        tokens:  [The, stranger, seemed, to, be, angry, at, ...]
        input:   [The, stranger, seemed, to, be]    (positions 0-4)
        target:  [stranger, seemed, to, be, angry]  (positions 1-5)
    
    This teaches the model: given "The" → predict "stranger",
    given "The stranger" → predict "seemed", etc.
    
    stride = max_length means NO OVERLAP between consecutive sequences,
    ensuring we cover the entire dataset without redundancy.
    """
    def __init__(self, tokens, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        for i in tqdm(range(0, len(tokens) - max_length, stride), desc="Creating pairs"):
            input_chunk = tokens[i : i + max_length]        # [i, i+4096)
            target_chunk = tokens[i + 1 : i + max_length + 1]  # [i+1, i+4097) — shifted!
            self.input_ids.append(torch.tensor(input_chunk, dtype=torch.int32))
            self.target_ids.append(torch.tensor(target_chunk, dtype=torch.int64))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


print("Creating train dataset...")
train_dataset = TextDataset(train_tokens, max_length=CONTEXT_LEN, stride=CONTEXT_LEN)
print("Creating validation dataset...")
val_dataset = TextDataset(val_tokens, max_length=CONTEXT_LEN, stride=CONTEXT_LEN)

print(f"\nTrain sequences: {len(train_dataset):,}")
print(f"Val sequences: {len(val_dataset):,}")

# DataLoader handles batching, shuffling, and parallel data loading
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)

del train_tokens, val_tokens
gc.collect()

## 4. GPT-OSS Architecture

### What's new vs GPT-2:
| Component | GPT-2 | GPT-OSS | Benefit |
|-----------|--------|---------|--------|
| Norm | LayerNorm | **RMSNorm** | ~15% faster, no mean subtraction |
| Position | Absolute (additive) | **RoPE** (rotation) | No semantic pollution |
| Attention | Full MHA | **GQA** (4:1) + **Sliding Window** | 4x smaller KV cache, O(n*w) |
| Noise handling | None | **Attention Sinks** | Absorbs excess attention |
| Activation | GELU | **SwiGLU** | Better gradients, gated flow |
| FFN | Single (4x expand) | **Mixture of Experts** | More capacity, sparse compute |

In [ ]:
@dataclass
class ModelConfig:
    """All architecture hyperparameters in one place.
    
    Full GPT-OSS 20B uses: hidden=2880, heads=64, kv_heads=8, experts=32, layers=24
    We scale down to fit on 1 GPU while keeping all architectural innovations.
    """
    num_hidden_layers: int = 12       # Number of transformer blocks (GPT-OSS 20B: 24)
    num_experts: int = 4             # Total MoE experts (GPT-OSS 20B: 32)
    experts_per_token: int = 2       # Active experts per token (GPT-OSS 20B: 4)
    vocab_size: int = 201088         # Harmony tokenizer vocabulary
    hidden_size: int = 1024          # Embedding dimension (GPT-OSS 20B: 2880)
    intermediate_size: int = 1024    # Expert FFN hidden dim
    swiglu_limit: float = 7.0        # Numerical clamp for SwiGLU stability
    head_dim: int = 64               # Dimension per attention head
    num_attention_heads: int = 16    # Query heads (GPT-OSS 20B: 64)
    num_key_value_heads: int = 4     # KV heads for GQA (ratio: 16/4 = 4:1)
    sliding_window: int = 128        # Window size for sliding attention layers
    initial_context_length: int = 4096  # Base context for RoPE frequency computation
    rope_theta: float = 150000.0     # RoPE base frequency (higher = slower decay)
    rope_scaling_factor: float = 1.0 # YaRN scaling (>1 extends context beyond training)
    rope_ntk_alpha: float = 1.0      # NTK-by-parts high frequency boundary
    rope_ntk_beta: float = 32.0      # NTK-by-parts low frequency boundary

In [ ]:
class RMSNorm(torch.nn.Module):
    """Root Mean Square Normalization (replaces LayerNorm).
    
    LayerNorm:  norm(x) = (x - mean) / std * gamma + beta
    RMSNorm:    norm(x) = x / RMS(x) * scale
    
    Why RMSNorm?
    - Skips mean subtraction → ~15% faster
    - Only has scale (no bias) → fewer parameters  
    - Empirically equal or better performance
    - Used in LLaMA, Gemma, GPT-OSS, and most modern LLMs
    
    The operation is per-vector (no cross-token interaction).
    Vector dimension stays the same: (seq, hidden) → (seq, hidden)
    """
    def __init__(self, num_features: int, eps: float = 1e-05, device=None):
        super().__init__()
        self.num_features = num_features
        self.eps = eps
        # Learnable scale parameter (initialized to 1 = identity)
        self.scale = torch.nn.Parameter(
            torch.ones(num_features, device=device, dtype=torch.float32)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Upcast to float32 for numerical precision during normalization
        t = x.float()
        # rsqrt = 1/sqrt — compute 1/RMS in one step
        # mean(t^2) = the "mean square", rsqrt gives 1/RMS
        t = t * torch.rsqrt(torch.mean(t**2, dim=-1, keepdim=True) + self.eps)
        # Apply learnable scale and cast back to original dtype (bfloat16)
        return (t * self.scale).to(x.dtype)

In [ ]:
def _apply_rotary_emb(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Apply rotation to pairs of dimensions.
    
    Standard 2D rotation matrix:
        [cos θ, -sin θ] [x1]   [x1*cos - x2*sin]
        [sin θ,  cos θ] [x2] = [x2*cos + x1*sin]
    
    This rotates each (x1, x2) pair by angle θ.
    θ depends on BOTH position AND dimension index.
    Rotation preserves vector magnitude (key advantage over additive position).
    """
    cos = cos.unsqueeze(-2).to(x.dtype)  # Broadcast across heads
    sin = sin.unsqueeze(-2).to(x.dtype)
    # Split vector into pairs: (x1, x2), (x3, x4), ...
    x1, x2 = torch.chunk(x, 2, dim=-1)
    # Apply 2D rotation to each pair
    o1 = x1 * cos - x2 * sin  # Rotated first half
    o2 = x2 * cos + x1 * sin  # Rotated second half
    return torch.cat((o1, o2), dim=-1)


class RotaryEmbedding(torch.nn.Module):
    """Rotary Positional Encodings (RoPE) with YaRN scaling.
    
    Instead of ADDING position vectors to token embeddings (GPT-2),
    we ROTATE the Q and K vectors in the attention mechanism.
    
    Why rotation?
    - Preserves vector magnitude (no semantic pollution)
    - Dot product Q·K naturally encodes RELATIVE position difference
    - Two tokens with same relative distance → same attention contribution
    
    Frequency design:
    - θ_i = 1 / (base ^ (2i / d))  where i = dimension index
    - Low indices (i small) → high frequency → capture fine word-order shifts
    - High indices (i large) → low frequency → preserve long-range context
    
    YaRN (scaling_factor > 1): extends context beyond training length
    by blending interpolation (high-freq dims) with extrapolation (low-freq dims).
    """
    def __init__(self, head_dim, base, dtype, initial_context_length=4096,
                 scaling_factor=1.0, ntk_alpha=1.0, ntk_beta=32.0, device=None):
        super().__init__()
        self.head_dim = head_dim
        self.base = base              # 150000 for GPT-OSS (10000 in original RoPE)
        self.dtype = dtype
        self.initial_context_length = initial_context_length
        self.scaling_factor = scaling_factor  # >1 activates YaRN context extension
        self.ntk_alpha = ntk_alpha
        self.ntk_beta = ntk_beta
        self.device = device

    def _compute_concentration_and_inv_freq(self):
        """Compute inverse frequencies for each dimension pair.
        
        With YaRN scaling (scaling_factor > 1):
        - High-frequency dims: interpolate (reduce freq to fit longer context)
        - Low-frequency dims: extrapolate (keep original freq)
        - Blend between the two using a smooth ramp
        See: https://arxiv.org/abs/2309.00071
        """
        # Base frequencies: base^(0/d), base^(2/d), base^(4/d), ...
        freq = self.base ** (
            torch.arange(0, self.head_dim, 2, dtype=torch.float, device=self.device)
            / self.head_dim
        )
        if self.scaling_factor > 1.0:
            # YaRN: scale attention to handle longer sequences
            concentration = 0.1 * math.log(self.scaling_factor) + 1.0
            d_half = self.head_dim / 2
            # NTK-by-parts: determine which dims to interpolate vs extrapolate
            low = (
                d_half * math.log(self.initial_context_length / (self.ntk_beta * 2 * math.pi))
                / math.log(self.base)
            )
            high = (
                d_half * math.log(self.initial_context_length / (self.ntk_alpha * 2 * math.pi))
                / math.log(self.base)
            )
            interpolation = 1.0 / (self.scaling_factor * freq)  # Scaled down for longer ctx
            extrapolation = 1.0 / freq                          # Original frequencies
            # Smooth ramp: 0 at low boundary, 1 at high boundary
            ramp = (torch.arange(d_half, dtype=torch.float32, device=freq.device) - low) / (high - low)
            mask = 1 - ramp.clamp(0, 1)
            # Blend: mask=1 → extrapolation, mask=0 → interpolation
            inv_freq = interpolation * (1 - mask) + extrapolation * mask
        else:
            # No scaling — standard RoPE
            concentration = 1.0
            inv_freq = 1.0 / freq
        return concentration, inv_freq

    def _compute_cos_sin(self, num_tokens):
        """Pre-compute cos/sin for all positions in the sequence."""
        concentration, inv_freq = self._compute_concentration_and_inv_freq()
        # positions: [0, 1, 2, ..., num_tokens-1]
        t = torch.arange(num_tokens, dtype=torch.float32, device=self.device)
        # Outer product: (num_tokens, d/2) — angle for each (position, dim_pair)
        freqs = torch.einsum("i,j->ij", t, inv_freq)
        cos = freqs.cos() * concentration
        sin = freqs.sin() * concentration
        return cos, sin

    def forward(self, query, key):
        """Apply RoPE to query and key vectors (NOT to values).
        
        Only Q and K are rotated because attention scores = Q·K^T.
        Rotating both means the dot product encodes relative position.
        Values carry content information — no position needed there.
        """
        num_tokens = query.shape[0]
        cos, sin = self._compute_cos_sin(num_tokens)
        # Reshape → apply rotation → reshape back
        query_shape = query.shape
        query = query.view(num_tokens, -1, self.head_dim)
        query = _apply_rotary_emb(query, cos, sin)
        query = query.reshape(query_shape)
        key_shape = key.shape
        key = key.view(num_tokens, -1, self.head_dim)
        key = _apply_rotary_emb(key, cos, sin)
        key = key.reshape(key_shape)
        return query, key

In [ ]:
def sdpa(Q, K, V, S, sm_scale, sliding_window=0):
    """Scaled Dot-Product Attention with GQA, sliding window, and attention sinks.
    
    This single function combines ALL attention innovations:
    1. GQA: K,V are broadcast across query groups (q_mult queries share 1 KV head)
    2. Causal mask: tokens cannot attend to future positions
    3. Sliding window: tokens can only look back `sliding_window` positions
    4. Attention sinks: learnable bias column absorbs excess attention noise
    
    Args:
        Q: (n_tokens, n_kv_heads, q_per_group, head_dim) — query vectors
        K: (n_tokens, n_kv_heads, head_dim) — key vectors (shared across group)
        V: (n_tokens, n_kv_heads, head_dim) — value vectors (shared across group)
        S: (n_attention_heads,) — learnable sink parameters, one per head
        sm_scale: 1/sqrt(head_dim) — prevents dot products from growing too large
        sliding_window: 0 = full attention, >0 = restricted lookback
    """
    n_tokens, n_heads, q_mult, d_head = Q.shape
    
    # GQA broadcast: expand K,V to match Q's grouped structure
    # Each KV head is shared by q_mult=4 query heads
    K = K[:, :, None, :].expand(-1, -1, q_mult, -1)  # (seq, kv_heads, q_mult, d)
    V = V[:, :, None, :].expand(-1, -1, q_mult, -1)
    
    # Reshape sinks: one learnable scalar per (head, query_in_group) pair
    S = S.reshape(n_heads, q_mult, 1, 1).expand(-1, -1, n_tokens, -1)
    
    # === MASKING ===
    # Causal mask: upper triangle = -inf (can't see future tokens)
    mask = torch.triu(Q.new_full((n_tokens, n_tokens), -float("inf")), diagonal=1)
    
    # Sliding window mask: lower triangle beyond window = -inf (can't see distant past)
    if sliding_window > 0:
        mask += torch.tril(
            mask.new_full((n_tokens, n_tokens), -float("inf")),
            diagonal=-sliding_window  # Anything more than `w` positions back is masked
        )
    
    # === ATTENTION SCORES ===
    # Q·K^T: compute attention score between every (query, key) pair
    QK = torch.einsum("qhmd,khmd->hmqk", Q, K)  # (heads, q_mult, seq, seq)
    QK *= sm_scale  # Scale by 1/sqrt(d) to prevent softmax saturation
    QK += mask[None, None, :, :]  # Apply causal + sliding mask
    
    # === ATTENTION SINKS ===
    # Append sink column: this extra column absorbs excess attention
    # that would otherwise inflate scores on irrelevant tokens
    QK = torch.cat([QK, S], dim=-1)  # (heads, q_mult, seq, seq+1)
    
    # Softmax over all positions INCLUDING the sink
    W = torch.softmax(QK, dim=-1)
    
    # Drop the sink column — it absorbed some probability, keeping real scores clean
    W = W[..., :-1]  # (heads, q_mult, seq, seq) — no longer sums to 1, but undiluted
    
    # === WEIGHTED SUM OF VALUES ===
    attn = torch.einsum("hmqk,khmd->qhmd", W, V)
    # Reshape from (seq, heads, q_mult, d) → (seq, total_head_dim)
    return attn.reshape(n_tokens, -1)

In [ ]:
class AttentionBlock(torch.nn.Module):
    """Full attention block: RMSNorm → QKV projection → RoPE → SDPA → Output projection → Residual.
    
    Key design choices:
    - Single fused QKV linear layer (more efficient than 3 separate)
    - GQA: 16 query heads share 4 KV heads (ratio 4:1)
    - Even layers use sliding window, odd layers use full causal attention
    - Learnable attention sinks per head
    - Residual connection: output = input + attention(norm(input))
    """
    def __init__(self, config: ModelConfig, layer_idx: int = 0, device=None):
        super().__init__()
        self.head_dim = config.head_dim
        self.num_attention_heads = config.num_attention_heads  # Q heads
        self.num_key_value_heads = config.num_key_value_heads  # KV heads (fewer!)
        
        # Alternating attention pattern:
        # Even layers → sliding window (local attention, efficient)
        # Odd layers → full causal (global context, expensive)
        self.sliding_window = config.sliding_window if layer_idx % 2 == 0 else 0
        
        # Attention sinks: one learnable bias per head
        # These absorb excess attention that deep layers pay to early tokens
        self.sinks = torch.nn.Parameter(
            torch.empty(config.num_attention_heads, device=device, dtype=torch.bfloat16)
        )
        
        # Pre-attention normalization (pre-norm architecture)
        self.norm = RMSNorm(config.hidden_size, device=device)
        
        # Fused QKV projection: one matrix multiply → split into Q, K, V
        # Q size: num_attention_heads * head_dim = 16*64 = 1024
        # K size: num_key_value_heads * head_dim = 4*64 = 256  (smaller! GQA saves here)
        # V size: same as K = 256
        # Total QKV: 1024 + 256 + 256 = 1536
        qkv_dim = config.head_dim * (
            config.num_attention_heads + 2 * config.num_key_value_heads
        )
        self.qkv = torch.nn.Linear(
            config.hidden_size, qkv_dim, device=device, dtype=torch.bfloat16
        )
        
        # Output projection: concatenated heads → hidden_size
        self.out = torch.nn.Linear(
            config.head_dim * config.num_attention_heads,  # 16*64 = 1024
            config.hidden_size,                             # 1024
            device=device, dtype=torch.bfloat16,
        )
        
        # Scaling factor: 1/sqrt(head_dim) prevents dot products from exploding
        self.sm_scale = 1 / math.sqrt(config.head_dim)  # 1/sqrt(64) = 0.125
        
        # RoPE: injects position info by rotating Q and K
        self.rope = RotaryEmbedding(
            config.head_dim, config.rope_theta, torch.float32,
            initial_context_length=config.initial_context_length,
            scaling_factor=config.rope_scaling_factor,
            ntk_alpha=config.rope_ntk_alpha,
            ntk_beta=config.rope_ntk_beta,
            device=device,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-norm: normalize BEFORE attention (more stable training)
        t = self.norm(x)
        
        # Single projection → split into Q, K, V
        qkv = self.qkv(t)
        q = qkv[:, : self.num_attention_heads * self.head_dim].contiguous()
        k = qkv[
            :,
            self.num_attention_heads * self.head_dim :
            (self.num_attention_heads + self.num_key_value_heads) * self.head_dim,
        ].contiguous()
        v = qkv[
            :,
            (self.num_attention_heads + self.num_key_value_heads) * self.head_dim :
            (self.num_attention_heads + 2 * self.num_key_value_heads) * self.head_dim,
        ].contiguous()
        
        # Reshape for GQA: group query heads with their shared KV head
        # Q: (seq, 4 kv_groups, 4 queries_per_group, 64 head_dim)
        # K: (seq, 4 kv_groups, 64 head_dim)
        # V: (seq, 4 kv_groups, 64 head_dim)
        q = q.view(-1, self.num_key_value_heads,
                   self.num_attention_heads // self.num_key_value_heads, self.head_dim)
        k = k.view(-1, self.num_key_value_heads, self.head_dim)
        v = v.view(-1, self.num_key_value_heads, self.head_dim)
        
        # Apply RoPE: rotate Q and K to encode position
        q, k = self.rope(q, k)
        
        # Full attention computation (GQA + masking + sinks)
        t = sdpa(q, k, v, self.sinks, self.sm_scale, self.sliding_window)
        
        # Output projection
        t = self.out(t)
        
        # Residual connection: prevents vanishing gradients in deep networks
        return x + t

In [ ]:
def swiglu(x, alpha: float = 1.702, limit: float = 7.0):
    """SwiGLU activation: Swish(gate_path) * (linear_path + 1)
    
    Evolution: ReLU → GLU → GELU → Swish → SwiGLU
    
    How it works:
    1. Split input into two interleaved streams (even/odd indices)
    2. Gate stream: apply Swish activation → x * sigmoid(alpha*x)
    3. Linear stream: keep as-is (plus bias of +1)
    4. Multiply them together → gated information flow
    
    Why +1 bias on linear path?
    Without it, if x_linear ≈ 0, output ≈ 0 regardless of gate.
    Adding 1 means the gate starts "open" and adjusts from there.
    
    Why better than GELU?
    - Two paths provide richer representation
    - Learned gating → selective feature amplification/suppression
    - Better gradient flow through the linear path
    - 2.7x fewer params per expert than traditional 4x-expand FFN
    
    Why clamp?
    Prevents overflow in sigmoid for very large values (numerical stability).
    """
    # Split into alternating channels: even indices = gate, odd = linear
    x_glu, x_linear = x[..., ::2], x[..., 1::2]
    
    # Clamp for numerical stability
    x_glu = x_glu.clamp(max=limit)
    x_linear = x_linear.clamp(-limit, limit)
    
    # Swish activation on gate path: x * sigmoid(alpha * x)
    # alpha=1.702 is empirically chosen (similar to GELU's Gaussian CDF)
    out_glu = x_glu * torch.sigmoid(alpha * x_glu)
    
    # Multiply gate output with (linear + 1) bias
    return out_glu * (x_linear + 1)


class MLPBlock(torch.nn.Module):
    """Mixture of Experts (MoE) block with SwiGLU activation.
    
    Instead of ONE large FFN (GPT-2: Linear→GELU→Linear),
    we have MULTIPLE small expert FFNs + a router that picks the best ones.
    
    Architecture per expert:
        Linear(hidden → 2*intermediate) → SwiGLU → Linear(intermediate → hidden)
    
    Router:
        Linear(hidden → num_experts) → top-K → softmax → weighted sum
    
    Why MoE?
    - 4 experts = 4x total capacity
    - Only 2 active per token = 50% actual compute
    - Different experts specialize (verbs, nouns, emotions, etc.)
    - At inference: 20B param model uses only 3.6B params per token
    """
    def __init__(self, config: ModelConfig, device=None):
        super().__init__()
        self.num_experts = config.num_experts
        self.experts_per_token = config.experts_per_token
        self.swiglu_limit = config.swiglu_limit
        self.world_size = dist.get_world_size() if dist.is_initialized() else 1
        
        # Pre-MoE normalization
        self.norm = RMSNorm(config.hidden_size, device=device)
        
        # Router: simple linear → scores for each expert
        # Each token gets a score for each expert; top-K are selected
        self.gate = torch.nn.Linear(
            config.hidden_size, config.num_experts, device=device, dtype=torch.bfloat16
        )
        
        # Expert FFNs: each is (Linear_up, Linear_down)
        # Linear_up: hidden → 2*intermediate (the 2x is for SwiGLU's two paths)
        # Linear_down: intermediate → hidden (contracts back)
        self.experts = torch.nn.ModuleList([
            torch.nn.Sequential(
                torch.nn.Linear(
                    config.hidden_size,
                    config.intermediate_size * 2 // self.world_size,
                    device=device, dtype=torch.bfloat16
                ),
                torch.nn.Linear(
                    config.intermediate_size // self.world_size,
                    config.hidden_size,
                    device=device, dtype=torch.bfloat16
                )
            ) for _ in range(config.num_experts)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len, hidden_size = x.shape
        t = self.norm(x)
        
        # ROUTING: compute expert scores for each token
        g = self.gate(t)  # (seq, num_experts) — each token scores each expert
        
        # Select top-K experts with highest scores
        experts = torch.topk(g, k=self.experts_per_token, dim=-1, sorted=True)
        # Normalize selected scores → blend weights that sum to 1
        expert_weights = torch.nn.functional.softmax(experts.values, dim=-1)
        expert_indices = experts.indices  # Which experts were selected
        
        t_flat = t.view(-1, hidden_size)
        expert_indices_flat = expert_indices.view(-1, self.experts_per_token)
        expert_weights_flat = expert_weights.view(-1, self.experts_per_token)
        output = torch.zeros_like(t_flat)

        # Process each expert: only tokens routed to it go through
        for expert_idx in range(self.num_experts):
            # Find which tokens selected this expert
            mask = (expert_indices_flat == expert_idx).any(dim=-1)
            if not mask.any():
                continue  # This expert is dormant — zero compute!
            
            token_indices = torch.where(mask)[0]
            expert_pos = (expert_indices_flat[token_indices] == expert_idx).nonzero(as_tuple=True)[1]
            expert_input = t_flat[token_indices]
            weights = expert_weights_flat[token_indices, expert_pos]
            
            # Forward through expert: Linear_up → SwiGLU → Linear_down
            expert_out = self.experts[expert_idx][0](expert_input)  # Expand
            expert_out = swiglu(expert_out, limit=self.swiglu_limit)  # Activate
            expert_out = self.experts[expert_idx][1](expert_out)      # Contract
            
            # Add weighted expert output to the token's result
            output[token_indices] += expert_out * weights.unsqueeze(-1)

        # For multi-GPU: aggregate outputs from all devices
        if self.world_size > 1:
            dist.all_reduce(output, op=dist.ReduceOp.SUM)
        
        output = output.view(seq_len, hidden_size)
        # Residual connection
        return x + output

In [ ]:
class TransformerBlock(torch.nn.Module):
    """One transformer block = Attention + MoE, both with residual connections.
    
    Data flow:
        x → AttentionBlock(x) → MLPBlock(x) → output
    
    Internally each sub-block does: output = input + sublayer(norm(input))
    This is the "pre-norm" architecture (norm before the layer, not after).
    """
    def __init__(self, config: ModelConfig, layer_idx: int, device=None):
        super().__init__()
        self.attn = AttentionBlock(config, layer_idx, device)
        self.mlp = MLPBlock(config, device)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.attn(x)  # Attention + residual
        x = self.mlp(x)   # MoE + residual
        return x


class GPToss(torch.nn.Module):
    """Complete GPT-OSS model.
    
    Forward pass:
        Token IDs (seq,)
        → Embedding (seq, hidden_size)          : convert IDs to dense vectors
        → 12x TransformerBlock (seq, hidden_size) : learn context + transform
        → RMSNorm (seq, hidden_size)            : final normalization
        → Unembedding (seq, vocab_size)          : project to vocabulary logits
    
    The unembedding layer outputs a score for EVERY token in the vocabulary.
    During training: cross-entropy loss between these logits and true next token.
    During inference: softmax → sample from distribution → next token.
    """
    def __init__(self, config: ModelConfig, device=None):
        super().__init__()
        self.config = config
        
        # Token embedding: (vocab_size × hidden_size) lookup table
        # Each of the 201088 tokens gets a learned 1024-d vector
        self.embedding = torch.nn.Embedding(
            config.vocab_size, config.hidden_size, device=device, dtype=torch.bfloat16
        )
        
        # Stack of transformer blocks (the "brain" of the model)
        # Even-indexed blocks: sliding window attention (local, fast)
        # Odd-indexed blocks: full causal attention (global, slower)
        self.blocks = torch.nn.ModuleList([
            TransformerBlock(config, layer_idx, device)
            for layer_idx in range(config.num_hidden_layers)
        ])
        
        # Final normalization before output projection
        self.norm = RMSNorm(config.hidden_size, device=device)
        
        # Unembedding: project hidden_size → vocab_size for next-token prediction
        # No bias — the logits are raw scores for each vocabulary token
        self.unembedding = torch.nn.Linear(
            config.hidden_size, config.vocab_size,
            bias=False, device=device, dtype=torch.bfloat16,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.embedding(x)        # (seq,) → (seq, 1024)
        for block in self.blocks:     # 12x: (seq, 1024) → (seq, 1024)
            x = block(x)
        x = self.norm(x)             # Final norm
        x = self.unembedding(x)      # (seq, 1024) → (seq, 201088)
        return x

## 5. Model Initialization

In [ ]:
config = ModelConfig(
    num_hidden_layers=12,
    num_experts=4,
    experts_per_token=2,
    vocab_size=201088,
    hidden_size=1024,
    intermediate_size=1024,
    head_dim=64,
    num_attention_heads=16,
    num_key_value_heads=4,
    sliding_window=128,
    initial_context_length=CONTEXT_LEN,
)

model = GPToss(config, device=DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params / 1e6:.1f}M")
print(f"Trainable parameters: {trainable_params / 1e6:.1f}M")
print(f"Model size (FP16): {total_params * 2 / 1e9:.2f} GB")
print(f"\nConfig: {asdict(config)}")

## 6. Training Loop

**Learning rate schedule:** Cosine annealing with linear warmup
- Warmup (0→200 steps): LR ramps 3e-5 → 3e-4 (explore safely while weights are random)
- Cosine decay (200→end): LR smoothly decays 3e-4 → 3e-5 (exploit good region)

**Optimizer:** AdamW with weight decay 0.1
- Adam tracks both mean and variance of gradients (adaptive per-param LR)
- Weight decay (L2 regularization) prevents overfitting
- β₂=0.95 (lower than default 0.999) — adapts faster for LLM training

In [ ]:
# === HYPERPARAMETERS ===
LEARNING_RATE = 3e-4    # Peak LR (reached after warmup)
MIN_LR = 3e-5           # Minimum LR (at end of cosine decay)
NUM_EPOCHS = 3          # Full passes through training data
WARMUP_STEPS = 200      # Steps to ramp LR from min to max
EVAL_FREQ = 200         # Evaluate on val set every N steps
EVAL_ITERS = 5          # Number of val batches per evaluation
WEIGHT_DECAY = 0.1      # L2 regularization strength
GRAD_CLIP = 1.0         # Max gradient norm (prevents exploding gradients)

# AdamW optimizer: Adam with decoupled weight decay
# betas=(0.9, 0.95) — standard for LLM training (faster β₂ adaptation)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.95), eps=1e-8
)

# Learning rate schedule: warmup → cosine decay
total_steps = NUM_EPOCHS * len(train_loader)
scheduler_warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_STEPS)
scheduler_decay = CosineAnnealingLR(optimizer, T_max=total_steps - WARMUP_STEPS, eta_min=MIN_LR)
scheduler = SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_decay], milestones=[WARMUP_STEPS])

print(f"Total training steps: {total_steps:,}")
print(f"Steps per epoch: {len(train_loader):,}")

In [ ]:
def compute_loss(input_batch, target_batch, model):
    """Compute cross-entropy loss for a batch of sequences.
    
    For each token position, the model outputs logits over the entire vocab (201088).
    Cross-entropy measures how far these logits are from the true next token.
    Lower loss = model is better at predicting the next token.
    """
    total_loss = 0
    for i in range(len(input_batch)):
        inp = input_batch[i].to(DEVICE, non_blocking=True)
        logits = model(inp)    # (seq_len, vocab_size) — score for every possible next token
        tgt = target_batch[i].to(DEVICE, non_blocking=True)
        # Cross-entropy: -log(P(correct_token)) averaged over all positions
        loss = F.cross_entropy(logits, tgt)
        total_loss += loss
    return total_loss / len(input_batch)


@torch.no_grad()
def evaluate(model, data_loader, num_batches=None):
    """Evaluate model on validation set (no gradient computation)."""
    model.eval()
    total_loss = 0.0
    num_batches = num_batches or len(data_loader)
    num_batches = min(num_batches, len(data_loader))
    for i, (inp, tgt) in enumerate(data_loader):
        if i >= num_batches:
            break
        loss = compute_loss(inp, tgt, model)
        total_loss += loss.item()
    model.train()
    return total_loss / num_batches


def generate_sample(model, prompt="Once upon a time", max_tokens=100):
    """Auto-regressive generation: predict one token at a time, append, repeat."""
    model.eval()
    idx = torch.tensor(tokenizer.encode(prompt), dtype=torch.int32).to(DEVICE)
    for _ in range(max_tokens):
        idx_cond = idx[-CONTEXT_LEN:]   # Keep within context window
        with torch.inference_mode():
            logits = model(idx_cond)
        logits = logits[-1, :] / 0.8    # Temperature: lower = more deterministic
        v, _ = torch.topk(logits, 50)   # Top-K: only consider 50 most likely tokens
        logits[logits < v[-1]] = -float('inf')
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)  # Sample from distribution
        idx = torch.cat((idx, idx_next), dim=0)
    model.train()
    return tokenizer.decode(idx.tolist())

In [ ]:
# Optional: Weights & Biases logging (set True to track experiments)
USE_WANDB = False

if USE_WANDB:
    import wandb
    wandb.init(
        project="nano-gpt-oss",
        name=f"gptoss-{total_params/1e6:.0f}M-h100",
        config={
            **asdict(config),
            "learning_rate": LEARNING_RATE,
            "num_epochs": NUM_EPOCHS,
            "batch_size": BATCH_SIZE,
            "context_length": CONTEXT_LEN,
            "total_params_M": total_params / 1e6,
        }
    )

In [ ]:
print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)

train_losses = []
val_losses = []
best_val_loss = float("inf")
global_step = 0
start_time = time.time()

model.train()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    epoch_steps = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for input_batch, target_batch in pbar:
        # === FORWARD PASS ===
        loss = compute_loss(input_batch, target_batch, model)
        
        # === BACKWARD PASS ===
        loss.backward()  # Compute gradients for all parameters
        
        # Gradient clipping: cap gradient magnitude to prevent explosions
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        
        # === PARAMETER UPDATE ===
        optimizer.step()              # Update weights using AdamW
        optimizer.zero_grad(set_to_none=True)  # Clear gradients for next step
        scheduler.step()              # Adjust learning rate

        epoch_loss += loss.item()
        epoch_steps += 1
        global_step += 1

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "lr": f"{optimizer.param_groups[0]['lr']:.2e}",
        })

        if USE_WANDB:
            wandb.log({"train/step_loss": loss.item(),
                       "lr": optimizer.param_groups[0]['lr']}, step=global_step)

        # === PERIODIC EVALUATION ===
        if global_step % EVAL_FREQ == 0:
            val_loss = evaluate(model, val_loader, num_batches=EVAL_ITERS)
            train_loss_avg = epoch_loss / epoch_steps
            train_losses.append(train_loss_avg)
            val_losses.append(val_loss)

            print(f"\n[Step {global_step:,}] Train loss: {train_loss_avg:.4f} | Val loss: {val_loss:.4f}")

            if USE_WANDB:
                wandb.log({"train/loss": train_loss_avg, "val/loss": val_loss}, step=global_step)

            # Save best model (lowest validation loss)
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save({
                    "model_state_dict": model.state_dict(),
                    "config": asdict(config),
                    "step": global_step,
                    "val_loss": val_loss,
                }, "checkpoints/gptoss_best.pt")
                print(f"  -> Saved best model (val_loss={val_loss:.4f})")

    # === END OF EPOCH ===
    # Save full checkpoint (can resume training from here)
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "config": asdict(config),
        "epoch": epoch,
        "global_step": global_step,
        "train_losses": train_losses,
        "val_losses": val_losses,
    }, f"checkpoints/gptoss_epoch{epoch+1}.pt")

    # Generate a sample to see qualitative progress
    sample = generate_sample(model, "A little girl named Lily")
    print(f"\n--- Sample (Epoch {epoch+1}) ---")
    print(sample[:500])
    print("---\n")

elapsed = time.time() - start_time
print(f"\nTraining complete in {elapsed/60:.1f} minutes")
print(f"Best validation loss: {best_val_loss:.4f}")

if USE_WANDB:
    wandb.finish()

## 7. Save Final Weights

In [ ]:
# Save final model with complete metadata for reproducibility
final_checkpoint = {
    "model_state_dict": model.state_dict(),
    "config": asdict(config),
    "total_params": total_params,
    "training_info": {
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "context_length": CONTEXT_LEN,
        "learning_rate": LEARNING_RATE,
        "final_train_loss": train_losses[-1] if train_losses else None,
        "best_val_loss": best_val_loss,
        "total_steps": global_step,
        "training_time_minutes": elapsed / 60,
    },
    "train_losses": train_losses,
    "val_losses": val_losses,
}

torch.save(final_checkpoint, "checkpoints/gptoss_final.pt")
print("Saved: checkpoints/gptoss_final.pt")

# Save config as JSON for easy inspection without loading PyTorch
with open("checkpoints/config.json", "w") as f:
    json.dump(asdict(config), f, indent=2)
print("Saved: checkpoints/config.json")

In [ ]:
# === SAVE TO GOOGLE DRIVE ===
# If running on Colab or a cloud instance with Drive access
SAVE_TO_DRIVE = True
DRIVE_PATH = "/content/drive/MyDrive/gptoss_weights"  # Adjust for your setup

if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        print("Not on Colab — saving to local path instead")

    os.makedirs(DRIVE_PATH, exist_ok=True)

    import shutil
    shutil.copy("checkpoints/gptoss_final.pt", f"{DRIVE_PATH}/gptoss_final.pt")
    shutil.copy("checkpoints/gptoss_best.pt", f"{DRIVE_PATH}/gptoss_best.pt")
    shutil.copy("checkpoints/config.json", f"{DRIVE_PATH}/config.json")

    print(f"\nWeights saved to: {DRIVE_PATH}")
    print(f"Files: {os.listdir(DRIVE_PATH)}")

## 8. Quick Inference Test

In [ ]:
test_prompts = [
    "Once upon a time, there was a little",
    "The big dog ran to the park and",
    "A brave knight went on a journey to",
    "Grandmother told the children a story about",
]

print("=" * 60)
print("INFERENCE SAMPLES")
print("=" * 60)

for prompt in test_prompts:
    output = generate_sample(model, prompt, max_tokens=150)
    print(f"\nPrompt: {prompt}")
    print(f"Output: {output[:300]}")
    print("-" * 40)